In [1]:
!uv add numpy
!uv add tensorflow
!uv add keras
!uv add seaborn
!uv add joblib
!uv add scikit-learn

Resolved 58 packages in 17ms
Checked 55 packages in 9ms
Resolved 58 packages in 4ms
Checked 55 packages in 3ms
Resolved 58 packages in 3ms
Checked 55 packages in 3ms
Resolved 58 packages in 4ms
Checked 55 packages in 3ms
Resolved 58 packages in 3ms
Checked 55 packages in 3ms
Resolved 58 packages in 4ms
Checked 55 packages in 3ms


In [2]:
import numpy as np
import os
import joblib
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (
    Embedding, LSTM, Bidirectional, Dense, Dropout,
    SpatialDropout1D, GlobalMaxPooling1D, Input, Conv1D
)
from tensorflow.keras.callbacks import (
    EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
)
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

In [3]:
import warnings
warnings.filterwarnings('ignore')

In [4]:
# for randomness
np.random.seed(42)
tf.random.set_seed(42)

In [5]:
os.makedirs('models', exist_ok=True)
os.makedirs('output', exist_ok=True)

In [6]:
# loading config
X_train = np.load('models/X_train_pad.npy')
X_val   = np.load('models/X_val_pad.npy')
X_test  = np.load('models/X_test_pad.npy')
y_train = np.load('models/y_train.npy')
y_val   = np.load('models/y_val.npy')
y_test  = np.load('models/y_test.npy')
class_weights = np.load('models/class_weights.npy')

config = joblib.load('models/config.pkl')
MAX_WORDS = config['MAX_WORDS']
MAX_LEN   = config['MAX_LEN']

class_weights_dict = {i: w for i, w in enumerate(class_weights)}

In [7]:
print(f"   X_train shape: {X_train.shape}")
print(f"   y_train shape: {y_train.shape}")
print(f"   Class weights: {class_weights_dict}")


   X_train shape: (53670, 100)
   y_train shape: (53670,)
   Class weights: {0: np.float64(1.0222273012970688), 1: np.float64(0.9922351636161952), 2: np.float64(0.9862726721428965)}


In [8]:
# lstm hyperparameters
EMBEDDING_DIM = 128
LSTM_UNITS_1  = 128
LSTM_UNITS_2  = 64
DENSE_UNITS   = 64
DROPOUT_RATE  = 0.3
NUM_CLASSES   = 3

In [10]:
# model architecture
model = Sequential([
    Embedding(
        input_dim=MAX_WORDS,
        output_dim=EMBEDDING_DIM,
        input_length=MAX_LEN,
        name='embedding'
    ),

    SpatialDropout1D(DROPOUT_RATE, name='spatial_dropout'),

    Bidirectional(
        LSTM(LSTM_UNITS_1, return_sequences=True, dropout=0.2, recurrent_dropout=0.1),
        name='bilstm_1'
    ),

    Bidirectional(
        LSTM(LSTM_UNITS_2, return_sequences=False, dropout=0.2, recurrent_dropout=0.1),
        name='bilstm_2'
    ),

    Dense(DENSE_UNITS, activation='relu', name='dense_1'),

    Dropout(DROPOUT_RATE, name='dropout_1'),

    Dense(NUM_CLASSES, activation='softmax', name='output')
])

# Print model summary (shows all layers and parameter counts)
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout                 │ ?                      │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bilstm_1 (Bidirectional)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bilstm_2 (Bidirectional)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [11]:
# compile model
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [12]:
# model callbacks
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),

    ModelCheckpoint(
        filepath='models/lstm_best_model.keras',
        monitor='val_loss',
        save_best_only=True,
        verbose=1
    ),

    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    ),
]

In [13]:
# train models
EPOCHS = 5
BATCH_SIZE = 64

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=class_weights_dict,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/5
839/839 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step - accuracy: 0.7403 - loss: 0.5784
Epoch 1: val_loss improved from None to 0.35107, saving model to models/lstm_best_model.keras

Epoch 1: finished saving model to models/lstm_best_model.keras
839/839 ━━━━━━━━━━━━━━━━━━━━ 62s 70ms/step - accuracy: 0.8121 - loss: 0.4541 - val_accuracy: 0.8600 - val_loss: 0.3511 - learning_rate: 0.0010
Epoch 2/5
839/839 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step - accuracy: 0.8765 - loss: 0.3269
Epoch 2: val_loss improved from 0.35107 to 0.32959, saving model to models/lstm_best_model.keras

Epoch 2: finished saving model to models/lstm_best_model.keras
839/839 ━━━━━━━━━━━━━━━━━━━━ 59s 70ms/step - accuracy: 0.8846 - loss: 0.3085 - val_accuracy: 0.8664 - val_loss: 0.3296 - learning_rate: 0.0010
Epoch 3/5
839/839 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step - accuracy: 0.9009 - loss: 0.2694
Epoch 3: val_loss did not improve from 0.32959
839/839 ━━━━━━━━━━━━━━━━━━━━ 59s 70ms/step - accuracy: 0.9068 - loss: 0.2576 - val_accurac

In [14]:
from tensorflow.keras.models import load_model
best_model = load_model('models/lstm_best_model.keras')

# model.evaluate() returns [loss, accuracy] on test data
test_loss, test_acc = best_model.evaluate(X_test, y_test, verbose=0)
print(f"   Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")
print(f"   Test Loss:     {test_loss:.4f}")

   Test Accuracy: 0.8710 (87.10%)
   Test Loss:     0.3251


In [43]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

tokenizer = joblib.load('models/tokenizer.pkl')
texts = ["bad don't buy it", "okayish", "Great product!", "needs improvement", "good experience overall"]
for text in texts:
  words = text.split()
  seq = tokenizer.texts_to_sequences(words)
  padded = pad_sequences(seq, maxlen=100, padding='post', truncating='post')

  predicted = best_model.predict(padded)
  predicted_class = np.argmax(predicted, axis=1)[0]

  labels = ['negative', 'neutral', 'positive']
  print(f"{text}: {labels[predicted_class]}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
bad don't buy it: negative
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
okayish: neutral
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
Great product!: positive
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
needs improvement: neutral
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
good experience overall: positive
